# Getting started

HERMESS simulates power system dynamics as one nonlinear
differential-algebraic system. This notebook runs a shipped system end to
end: pick a case, simulate it, read the initial power flow, and plot the
trajectories.

In [ ]:
import matplotlib.pyplot as plt

import hermess

hermess.list_systems()

`3bus` is the smallest shipped system that contains both kinds of dynamic
device: a Sauer-Pai synchronous machine at bus 1, a grid-forming converter at
bus 3 and a ZIP load at bus 2. Its disturbance file opens the line between
buses 3 and 1 at t = 3 s.

The shipped case pairs a dynamic LCL output filter with the dynamic network
model, so we keep `line_dyn=True` and integrate with a 0.1 ms step.
`simulate` accepts any field of the configuration as a keyword argument and
returns the finished model; `extract_results` turns it into plain numpy
containers.

In [ ]:
dae = hermess.simulate("3bus", T_end=6.0, ts=1e-4, line_dyn=True)
res = hermess.extract_results(dae)

The run is initialized from a power flow, so t = 0 s is a steady state:

In [ ]:
res.power_flow_bus

`res.voltage` maps each bus to its complex voltage trajectory. The line
opening at t = 3 s and the electromagnetic transient it excites are both
visible:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
for bus in res.buses():
    ax.plot(res.t, res.voltage_magnitude(bus), lw=0.8, label=f"bus {bus}")
ax.axvline(3.0, color="0.6", ls=":", lw=1)
ax.set_xlabel("t [s]")
ax.set_ylabel("|v| [p.u.]")
ax.legend()
fig.tight_layout()

Every device keeps its own differential states, listed per unit in
`res.devices`:

In [ ]:
for dev in res.devices:
    print(f"{dev.unit} at bus {dev.bus}: {', '.join(dev.states)}")

The machine speed and the converter's filtered active power tell the
electromechanical story of the event: the converter picks up the power the
lost line no longer carries, and the machine settles at a slightly different
angle.

In [ ]:
sg1 = next(d for d in res.devices if d.unit == "SG1")
gfm = next(d for d in res.devices if d.unit == "GFMI2")

fig, axes = plt.subplots(2, 1, figsize=(8, 4.6), sharex=True)
axes[0].plot(res.t, sg1.states["omega"], color="#215CAF")
axes[0].set_ylabel(r"$\omega_{\mathrm{SG1}}$ [p.u.]")
axes[1].plot(res.t, gfm.states["Pc_tilde"], color="#007894")
axes[1].set_ylabel(r"$\tilde p_{c,\mathrm{GFM}}$ [p.u.]")
axes[1].set_xlabel("t [s]")
for ax in axes:
    ax.axvline(3.0, color="0.6", ls=":", lw=1)
fig.tight_layout()

From here, each of the remaining notebooks picks out one aspect of the
simulator: scheduled disturbances, the hybrid EMT/RMS network, small-signal
analysis and parametric sensitivities. The user guide describes the system
files, and the model library documents every device class.